In [1]:
import os
import math
import cadquery as cq
from ocp_vscode import show

In [2]:
# ==============================================================================
# 1. PARAMETERS (9.25mm x 1.90mm Stick)
# ==============================================================================
stick_width = 9.25      # Broad face
stick_thickness = 1.90  # Thin edge
clearance = 0.25        # Fit clearance

pocket_thick = stick_thickness + clearance  # 2.15 mm flex gap
pocket_width = stick_width + clearance      # 9.50 mm cavity width

wall_thickness = 1.2    # Uniform 1.2mm wall thickness
straight_height = 9.0   # Enclosed pocket depth
curved_height = 7.0     # Upper arm flex length
base_thickness = 3.0    # Solid base floor

# Increased inward bend for over-center snap-fit (0.55mm over-bend)
inward_bend = 0.55      
saddle_depth = 0.8      # Mode 2 top-wall alignment notch depth

pocket_wall = wall_thickness
clamp_depth = pocket_width + (2 * pocket_wall)  # 11.90 mm overall span

gap_width = pocket_thick                        # 2.15 mm
half_gap = gap_width / 2.0                     # 1.075 mm
outer_x = half_gap + wall_thickness             # 2.275 mm
mid_y = base_thickness + straight_height       # 12.0 mm (top of 9mm pocket)

# ==============================================================================
# 2. CONCENTRIC ARC MATH (Over-Center Snap Arms)
# ==============================================================================
R_o = (curved_height**2 + inward_bend**2) / (2.0 * inward_bend)
R_i = R_o - wall_thickness

x_c = outer_x - R_o
y_c = mid_y

sin_theta = curved_height / R_o
cos_theta = 1.0 - (inward_bend / R_o)

outer_tip_x = x_c + R_o * cos_theta
outer_tip_y = y_c + R_o * sin_theta

inner_tip_x = x_c + R_i * cos_theta
inner_tip_y = y_c + R_i * sin_theta

cos_half = math.sqrt((1.0 + cos_theta) / 2.0)
sin_half = math.sqrt((1.0 - cos_theta) / 2.0)

outer_mid_x = x_c + R_o * cos_half
outer_mid_y = y_c + R_o * sin_half

inner_mid_x = x_c + R_i * cos_half
inner_mid_y = y_c + R_i * sin_half

# Over-center snap lip profile at tip
tip_center_x = (outer_tip_x + inner_tip_x) / 2.0
tip_center_y = (outer_tip_y + inner_tip_y) / 2.0
tip_radius = wall_thickness / 2.0

tip_apex_x = tip_center_x - tip_radius * sin_theta
tip_apex_y = tip_center_y + tip_radius * cos_theta

# ==============================================================================
# 3. SOLID GENERATION
# ==============================================================================
# Main U-profile with snap-over arm tips
half_clamp = (
    cq.Workplane("XY")
    .moveTo(0, 0)
    .lineTo(outer_x, 0)
    .lineTo(outer_x, mid_y)
    # Outer arc
    .threePointArc((outer_mid_x, outer_mid_y), (outer_tip_x, outer_tip_y))
    # Tangent snap tip arc
    .threePointArc((tip_apex_x, tip_apex_y), (inner_tip_x, inner_tip_y))
    # Inner arc
    .threePointArc((inner_mid_x, inner_mid_y), (half_gap, mid_y))
    .lineTo(half_gap, base_thickness)
    .lineTo(0, base_thickness)
    .close()
    .extrude(clamp_depth)
)

u_clamp = half_clamp.union(half_clamp.mirror("YZ"))

# Front and back 1.2mm walls enclosing the lower 9.0mm pocket cavity
pocket_center_y = base_thickness + (straight_height / 2.0)

back_wall = (
    cq.Workplane("XY")
    .moveTo(0, pocket_center_y)
    .rect(gap_width, straight_height)
    .extrude(pocket_wall)
)

front_wall = (
    cq.Workplane("XY")
    .workplane(offset=clamp_depth - pocket_wall)
    .moveTo(0, pocket_center_y)
    .rect(gap_width, straight_height)
    .extrude(pocket_wall)
)

pocket_walls = back_wall.union(front_wall)

# Mode 2 Alignment Saddle: 0.8mm deep notch cut into the top edge of front/back walls
saddle_cut = (
    cq.Workplane("XY")
    .moveTo(0, mid_y - (saddle_depth / 2.0))
    .rect(pocket_thick, saddle_depth)
    .extrude(clamp_depth)
)

pocket_walls = pocket_walls.cut(saddle_cut)

# Combine elements into finished dual-mode connector
result = u_clamp.union(pocket_walls)

# Lead-in chamfers at upper entry points
result = result.faces(">Z").edges().chamfer(0.25)

# Render the solid
show(result)

Using port 3939
+
